[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janperler9-dotcom/EAIF-AS26/blob/main/notebooks/Python/05_EAIF_Lung_Images_CNN_vs_Boosting.ipynb)

# Practical 4 · Smoking status from chest images: CNN vs. gradient boosting

**Setting.** A Swiss insurer underwrites supplementary health insurance (VVG/LCA). Unlike basic insurance (KVG/LAMal), it may decline applicants or exclude conditions, and it asks about smoking. Suppose the underwriter has a recent chest radiograph of the applicant. Can a model tell from the image whether the applicant

* **never** smoked,
* is a **former** smoker (quit within the last 5 years), or
* is a **current** smoker?

**Data.** `data/lung_xray/` holds 60 synthetic grey-scale chest images (64 × 64 pixels, 20 per class) and `annotations.csv` with the label of each image. The images are schematic, not real radiographs: smokers show more small bright spots and streaks inside the lungs (a "dirty chest"), and every image is randomly rotated, scaled and shifted, as if the patient stood differently in front of the detector.

**Plan.**
1. Load and explore the images.
2. Set up an honest evaluation for a tiny data set (balanced splits, several metrics, repeated splits).
3. Model A: gradient boosting on a **table of pixels** (one column per pixel).
4. Model B: a small **convolutional network (CNN)** that sees the image as a grid.
5. Compare them over 10 random splits.
6. Explain the gap with controlled experiments (shifting images, implanting lesions, class activation maps).
7. Give boosting a fair chance with **engineered features**.
8. Discuss whether an insurer may use such a model in Switzerland.

*Runtime:* about 10 minutes on the Colab CPU (Section 5 trains 20 networks), faster with a GPU runtime (Runtime → Change runtime type → GPU).

In [ ]:
import time, urllib.request
from pathlib import Path

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi
import torch, torch.nn as nn, torch.nn.functional as F

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, recall_score, log_loss,
                             roc_auc_score, confusion_matrix)

SEED = 2026
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

CLASSES = ["never", "former", "current"]
NAMES = {"never": "never smoked", "former": "quit < 5 years", "current": "smokes now"}
COLORS = {"never": "#1A3F99", "former": "#E0A800", "current": "#C8553D"}
pd.set_option("display.precision", 3)

## 1 · Load and explore the images

The files are read directly from the course GitHub repository, so the notebook runs in Colab without uploading anything.

In [ ]:
RAW = "https://raw.githubusercontent.com/janperler9-dotcom/EAIF-AS26/main/data/lung_xray/"
LOCAL = Path("lung_xray"); LOCAL.mkdir(exist_ok=True)

ann = pd.read_csv(RAW + "annotations.csv")
for f in ann.file:
    if not (LOCAL / f).exists():
        urllib.request.urlretrieve(RAW + f, LOCAL / f)

X = np.stack([np.asarray(Image.open(LOCAL / f).convert("L"), dtype=np.float32) / 255 for f in ann.file])
y = ann.smoking_status.map({c: k for k, c in enumerate(CLASSES)}).to_numpy()

print("images:", X.shape, "| intensity range:", X.min().round(2), "to", X.max().round(2))
print(ann.smoking_status.value_counts())
ann.head()

In [ ]:
fig, axs = plt.subplots(3, 6, figsize=(12, 6.4))
for k, c in enumerate(CLASSES):
    for j, i in enumerate(np.where(y == k)[0][:6]):
        axs[k, j].imshow(X[i], cmap="gray", vmin=0, vmax=1); axs[k, j].axis("off")
        axs[k, j].set_title(ann.image_id[i], fontsize=8)
    axs[k, 0].text(-8, 32, NAMES[c], rotation=90, va="center", ha="center", fontsize=10, color=COLORS[c])
plt.suptitle("Six images per class (rows)", fontsize=11); plt.tight_layout(); plt.show()

**What we see.** Every image shows a body outline, two dark lungs, a spine and faint ribs. Smokers' lungs contain more small bright spots and a few thin streaks. The chests are also tilted, enlarged or shifted by different amounts, so the same anatomical point sits at different pixel positions in different images.

### 1.1 · Average image per class
If the class signal sat at fixed pixel positions, the class averages would differ clearly at those positions. Because the spots appear at random places, averaging smears them out.

In [ ]:
means = np.stack([X[y == k].mean(0) for k in range(3)])
fig, axs = plt.subplots(1, 4, figsize=(13, 3.4))
for k, c in enumerate(CLASSES):
    axs[k].imshow(means[k], cmap="gray", vmin=0, vmax=1); axs[k].set_title(f"mean · {NAMES[c]}", fontsize=10)
d = means[2] - means[0]; lim = np.abs(d).max()
im = axs[3].imshow(d, cmap="RdBu_r", vmin=-lim, vmax=lim); axs[3].set_title("mean(current) − mean(never)", fontsize=10)
plt.colorbar(im, ax=axs[3], fraction=0.046)
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()
print(f"largest pixel-wise difference of class means: {lim:.3f} (on a 0-1 scale)")

The averages look almost identical. The difference map shows only a faint, diffuse brightening of the lung fields for smokers: the spots are there, but at a different place in every image. **A model that treats each pixel as a separate column therefore sees a weak and noisy signal in each column.** This is the core problem of the pixel table.

### 1.2 · Two representations of the same image

In [ ]:
X_tab = X.reshape(len(X), -1)          # table: one row per image, one column per pixel
X_grid = X[:, None]                    # grid: (images, channels, height, width) for a CNN
print("table for boosting:", X_tab.shape, " -> 4,096 columns, neighbourhood information lost")
print("grid for the CNN:  ", X_grid.shape, " -> pixel neighbourhoods kept")

In the table, column 1,000 and column 1,001 are just two unrelated features for the trees; the model does not know they are neighbours. A CNN slides the same small filter over every position, so a "bright spot detector" learned in one place works everywhere.

## 2 · Evaluation design for 60 images

* **Balanced splits.** Each test set has 18 images, 6 per class (`StratifiedShuffleSplit`), and each training set has 42.
* **Several metrics.** Accuracy (chance = 1/3), macro-F1, recall per class, log-loss (always answering 1/3 gives ln 3 ≈ 1.10, so anything above that is worse than guessing in probability terms), and one-vs-rest macro AUC.
* **Repetition.** With 18 test images, one image is 5.6 percentage points of accuracy. A single split is therefore mostly noise, so we repeat over 10 splits and report mean ± standard deviation.

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=10, test_size=18, random_state=SEED)
SPLITS = list(splitter.split(X_tab, y))
tr, te = SPLITS[0]                      # the "main" split used for the experiments below
print("train per class:", np.bincount(y[tr]), "| test per class:", np.bincount(y[te]))

def evaluate(y_true, proba):
    """All metrics for one test set; proba has shape (n, 3)."""
    p = np.clip(proba, 1e-6, 1); p = p / p.sum(1, keepdims=True)
    pred = p.argmax(1)
    rec = recall_score(y_true, pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"accuracy": accuracy_score(y_true, pred),
            "macro_F1": f1_score(y_true, pred, average="macro"),
            "recall_never": rec[0], "recall_former": rec[1], "recall_current": rec[2],
            "log_loss": log_loss(y_true, p, labels=[0, 1, 2]),
            "AUC": roc_auc_score(y_true, p, multi_class="ovr", labels=[0, 1, 2])}

## 3 · Model A: gradient boosting on 4,096 pixel columns

With only 42 training rows, we lower `min_samples_leaf` (default 20, which would allow at most two leaves) and add some L2 regularisation.

In [ ]:
def fit_boosting(Xtr, ytr):
    m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_leaf_nodes=15,
                                       min_samples_leaf=3, l2_regularization=1.0, random_state=SEED)
    return m.fit(Xtr.reshape(len(Xtr), -1), ytr)

def predict_boosting(m, Xs):
    return m.predict_proba(Xs.reshape(len(Xs), -1))

t0 = time.time()
gb = fit_boosting(X[tr], y[tr])
p_gb = predict_boosting(gb, X[te])
print(f"trained in {time.time() - t0:.1f}s")
pd.Series(evaluate(y[te], p_gb), name="boosting · pixels (main split)")

### 3.1 · Which pixels does boosting use?
We count how often each pixel was used to split in all trees. If boosting had learned "spots anywhere in the lung", the splits would spread over the whole lung area; if it memorised positions, they would concentrate on a few scattered pixels.

In [ ]:
def split_usage(model, n_pix=64 * 64):
    counts = np.zeros(n_pix)
    for trees in model._predictors:          # one list of trees per boosting iteration
        for tree in trees:
            nodes = tree.nodes[~tree.nodes["is_leaf"].astype(bool)]
            np.add.at(counts, nodes["feature_idx"], 1)
    return counts.reshape(64, 64)

use = split_usage(gb)
fig, axs = plt.subplots(1, 2, figsize=(8, 3.6))
axs[0].imshow(means.mean(0), cmap="gray"); axs[0].set_title("average chest", fontsize=10)
axs[1].imshow(means.mean(0), cmap="gray"); axs[1].imshow(np.ma.masked_equal(use, 0), cmap="autumn", alpha=.9)
axs[1].set_title(f"pixels used for splits ({int((use > 0).sum())} of 4,096)", fontsize=10)
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

*Note:* `_predictors` is an internal attribute of scikit-learn. If it changes in a future version, simply skip this cell.

The trees split on a few hundred scattered pixels. Each split asks "is pixel (r, c) brighter than t?", a question that matters only if a spot happens to land on exactly that pixel in a new image. With random lesion positions and a moving chest, such rules rarely transfer.

## 4 · Model B: a small convolutional network

**Architecture.** Three blocks of 3 × 3 convolutions (16 → 32 → 32 filters), with max pooling between them, then **global average pooling** and one linear layer.
* Convolutions share their weights across positions, so one filter detects a bright spot wherever it is (translation equivariance).
* Global average pooling averages each filter map over the image. The resulting number grows with **how many** spots the image contains, which is exactly what separates the three classes. Max pooling would only record whether at least one spot exists.
* There are about 25,000 parameters, many for 42 images. We counter this with weight decay and **data augmentation**: each training batch is randomly rotated (±12°), scaled (±10 %), sheared, shifted (±4 px) and mirrored, which are label-preserving changes that mimic how patients stand.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                                 nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU())
        self.features = nn.Sequential(block(1, 16), nn.MaxPool2d(2),     # 64 -> 32
                                      block(16, 32), nn.MaxPool2d(2),    # 32 -> 16
                                      block(32, 32))                     # 32 maps of 16 x 16
        self.head = nn.Linear(32, n_classes)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3)))             # global average pooling

print("parameters:", sum(p.numel() for p in SmallCNN().parameters()))


def augment_batch(xb):
    """Random rotation, scale, shear, shift and horizontal flip for a batch (n, 1, 64, 64)."""
    n, dev = xb.shape[0], xb.device
    u = lambda: torch.rand(n, device=dev) * 2 - 1               # uniform in [-1, 1]
    ang, sc, sh = u() * np.deg2rad(12), 1 + 0.1 * u(), 0.08 * u()
    flip = torch.where(torch.rand(n, device=dev) < 0.5, -1.0, 1.0)
    theta = torch.zeros(n, 2, 3, device=dev)
    theta[:, 0, 0] = torch.cos(ang) * sc * flip
    theta[:, 0, 1] = -torch.sin(ang) * sc + sh
    theta[:, 1, 0] = torch.sin(ang) * sc * flip
    theta[:, 1, 1] = torch.cos(ang) * sc
    theta[:, :, 2] = 0.125 * torch.stack([u(), u()], 1)          # shift of up to 4 of 32 half-widths
    grid = F.affine_grid(theta, list(xb.shape), align_corners=False)
    out = F.grid_sample(xb, grid, padding_mode="border", align_corners=False)
    return out + 0.01 * torch.randn_like(out)


def train_cnn(Xtr, ytr, augment=True, epochs=150, seed=SEED):
    torch.manual_seed(seed)
    model = SmallCNN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    xt = torch.tensor(Xtr[:, None], dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
    for _ in range(epochs):
        model.train()
        for b in torch.randperm(len(xt), device=DEVICE).split(16):
            xb = augment_batch(xt[b]) if augment else xt[b]
            loss = F.cross_entropy(model(xb), yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    return model.eval()


def predict_cnn(model, Xs):
    model.eval()
    with torch.no_grad():
        xs = torch.tensor(np.asarray(Xs)[:, None], dtype=torch.float32, device=DEVICE)
        return F.softmax(model(xs), 1).cpu().numpy()

In [ ]:
# What the augmentation does to one image
xb = torch.tensor(np.repeat(X[tr[:1], None], 6, 0))
aug = augment_batch(xb).numpy()[:, 0]
fig, axs = plt.subplots(1, 7, figsize=(13, 2.2))
axs[0].imshow(X[tr[0]], cmap="gray", vmin=0, vmax=1); axs[0].set_title("original", fontsize=9)
for j in range(6):
    axs[j + 1].imshow(aug[j], cmap="gray", vmin=0, vmax=1); axs[j + 1].set_title(f"augmented {j + 1}", fontsize=9)
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
t0 = time.time()
cnn = train_cnn(X[tr], y[tr], augment=True)
p_cnn = predict_cnn(cnn, X[te])
print(f"trained in {time.time() - t0:.1f}s")
pd.DataFrame({"boosting · pixels": evaluate(y[te], p_gb), "CNN": evaluate(y[te], p_cnn)})

## 5 · Fair comparison over 10 splits

We compare four models on the same 10 splits:

| model | input | purpose |
|---|---|---|
| boosting · pixels | 4,096 columns | the naive baseline |
| boosting · pixels + aug. | 4,096 columns, each training image plus 10 augmented copies | can augmentation alone fix boosting? |
| CNN without aug. | grid | what does the architecture alone contribute? |
| CNN | grid + augmentation | the full model |

Engineered features for boosting follow in Section 7.

In [ ]:
def augment_np(img, rng):
    """NumPy version of the augmentation, used to enlarge the boosting training table."""
    ang, sc, sh = np.deg2rad(rng.uniform(-12, 12)), rng.uniform(0.9, 1.1), rng.uniform(-0.08, 0.08)
    A = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]]) @ np.array([[1, sh], [0, 1]]) / sc
    c = np.array([31.5, 31.5]); off = c - A @ c + rng.uniform(-4, 4, 2)
    out = ndi.affine_transform(img, A, offset=off, order=1, mode="nearest")
    return out[:, ::-1] if rng.random() < 0.5 else out

def fit_boosting_aug(Xtr, ytr, n_copies=10, seed=SEED):
    rng = np.random.default_rng(seed)
    Xa = np.concatenate([Xtr] + [np.stack([augment_np(im, rng) for im in Xtr]) for _ in range(n_copies)])
    ya = np.tile(ytr, n_copies + 1)
    return fit_boosting(Xa, ya)

MODELS = {
    "boosting · pixels":        (fit_boosting,                                   predict_boosting),
    "boosting · pixels + aug.": (fit_boosting_aug,                               predict_boosting),
    "CNN without aug.":         (lambda a, b: train_cnn(a, b, augment=False),    predict_cnn),
    "CNN":                      (lambda a, b: train_cnn(a, b, augment=True),     predict_cnn),
}

def run_splits(models, splits=SPLITS):
    rows, pooled = [], {m: [] for m in models}
    for s, (tr_, te_) in enumerate(splits):
        for name, (fit, pred) in models.items():
            proba = pred(fit(X[tr_], y[tr_]), X[te_])
            rows.append({"split": s, "model": name, **evaluate(y[te_], proba)})
            pooled[name].append((y[te_], proba))
        print(f"split {s + 1}/{len(splits)} done")
    return pd.DataFrame(rows), pooled

t0 = time.time()
res, pooled = run_splits(MODELS)
print(f"total time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
order = list(MODELS)
summary = res.groupby("model").agg(["mean", "std"]).drop(columns="split").loc[order]
display(summary.round(3))

acc = res.pivot(index="split", columns="model", values="accuracy")
print("Splits in which the CNN beats boosting on pixels:",
      int((acc["CNN"] > acc["boosting · pixels"]).sum()), "of", len(acc))

fig, ax = plt.subplots(figsize=(8, 3.6))
for j, m in enumerate(order):
    ax.scatter(np.full(len(acc), j) + np.random.uniform(-.08, .08, len(acc)), acc[m], alpha=.7)
    ax.hlines(acc[m].mean(), j - .25, j + .25, color="black")
ax.axhline(1 / 3, ls="--", color="grey"); ax.text(len(order) - .5, 1 / 3 + .01, "chance", color="grey", ha="right")
ax.set_xticks(range(len(order)), order, fontsize=9); ax.set_ylabel("test accuracy"); ax.set_ylim(0, 1.05)
ax.set_title("Accuracy per split (dots) and mean (bar)", fontsize=10); plt.tight_layout(); plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for a, m in zip(axs, ["boosting · pixels", "CNN"]):
    yt_ = np.concatenate([t for t, _ in pooled[m]]); pr_ = np.concatenate([p for _, p in pooled[m]]).argmax(1)
    cm = confusion_matrix(yt_, pr_, labels=[0, 1, 2], normalize="true")
    a.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            a.text(j, i, f"{cm[i, j]:.0%}", ha="center", va="center", color="white" if cm[i, j] > .5 else "black")
    a.set_xticks(range(3), [NAMES[c] for c in CLASSES], fontsize=8, rotation=15)
    a.set_yticks(range(3), [NAMES[c] for c in CLASSES], fontsize=8)
    a.set_xlabel("predicted"); a.set_ylabel("true"); a.set_title(f"{m} · {len(yt_)} pooled test images", fontsize=10)
plt.tight_layout(); plt.show()

**Interpretation** (check the numbers against your own run; they vary slightly with library versions and hardware):
* **Boosting on pixels** should land well above chance (0.33) but far from useful, with large swings between splits. Look at its log-loss: if it exceeds ln 3 ≈ 1.10, the model is *confidently wrong*, worse in probability terms than always answering 1/3. Its AUC above 0.5 shows some ranking signal, mostly the faint overall brightening of smokers' lungs from Section 1.1.
* **Augmentation helps boosting only partly.** The copies teach the trees that the chest moves, but a tree can still only ask about single pixels, so it cannot express "a bright spot somewhere in the lung".
* **The CNN** should be clearly better on every metric and win in (almost) all splits. Its errors concentrate on the middle class: a recent quitter with few lesions looks like a never-smoker, and a heavy former smoker looks like a current one. This is the expected confusion, because the classes differ only in **how many** lesions there are.
* **CNN without augmentation vs. with:** convolutions plus global average pooling already handle *where* a lesion is. Augmentation adds robustness to rotation and scale of the whole chest, which convolutions do not provide by themselves.
* The standard deviations are large. Only the repetition over 10 splits makes the ranking credible; a single 18-image test set could easily reverse a small difference.

## 6 · Why does the CNN win? Three experiments

### 6.1 · Shift test images by a few pixels
A patient standing a few centimetres to the left moves every pixel. We shift the test images of the main split by *k* pixels in a random direction and re-score both models (trained on the main split above).

In [ ]:
rng = np.random.default_rng(SEED)
shifts, rows = [0, 2, 4, 6, 8], []
for k in shifts:
    for rep in range(5):
        ang = rng.uniform(0, 2 * np.pi, len(te))
        Xs = np.stack([ndi.shift(X[i], (k * np.sin(a), k * np.cos(a)), order=1, mode="nearest") for i, a in zip(te, ang)])
        rows.append({"shift": k, "boosting · pixels": accuracy_score(y[te], predict_boosting(gb, Xs).argmax(1)),
                     "CNN": accuracy_score(y[te], predict_cnn(cnn, Xs).argmax(1))})
shift_res = pd.DataFrame(rows).groupby("shift").mean()
display(shift_res.round(3))
shift_res.plot(marker="o", figsize=(6, 3.4), ylim=(0, 1.05), ylabel="test accuracy", xlabel="shift in pixels",
               title="Robustness to shifting the whole image"); plt.axhline(1 / 3, ls="--", c="grey"); plt.show()

Boosting relies on fixed pixel positions, so its accuracy should fall as the shift grows. The CNN's filters move with the image, and global pooling ignores where the response occurs, so it should stay roughly flat.

### 6.2 · Implant lesions into never-smoker images
This experiment separates "spots" from "overall brightness". We take the never-smoker test images, add 0 to 12 artificial bright nodules at random positions **inside the lungs**, and measure how the predicted probability of *smokes now* changes. The lung region is found automatically: dark areas inside the body that do not touch the image border.

In [ ]:
def lung_mask(img):
    dark = ndi.gaussian_filter(img, 1.5) < 0.33
    lab, n = ndi.label(dark)
    border = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
    mask = np.isin(lab, np.setdiff1d(np.arange(1, n + 1), border))
    mask = ndi.binary_fill_holes(ndi.binary_closing(mask, iterations=2))
    if mask.sum() < 50:                                    # fallback: central region
        mask = np.zeros_like(img, bool); mask[16:52, 10:54] = True
    return mask

def implant(img, n_spots, rng, amp=0.4):
    out, m = img.copy(), lung_mask(img)
    yy, xx = np.mgrid[0:64, 0:64]; ly, lx = np.where(m)
    for _ in range(n_spots):
        i = rng.integers(len(ly)); r = rng.uniform(1.8, 2.6)
        out += amp * np.exp(-(((yy - ly[i]) / r) ** 2 + ((xx - lx[i]) / r) ** 2)) * m
    return np.clip(out, 0, 1).astype(np.float32)

never_te = te[y[te] == 0]
fig, axs = plt.subplots(1, 4, figsize=(9, 2.6))
axs[0].imshow(X[never_te[0]], cmap="gray", vmin=0, vmax=1); axs[0].set_title("never smoker", fontsize=9)
axs[1].imshow(lung_mask(X[never_te[0]]), cmap="gray"); axs[1].set_title("detected lungs", fontsize=9)
for a, n in zip(axs[2:], [4, 10]):
    a.imshow(implant(X[never_te[0]], n, np.random.default_rng(1)), cmap="gray", vmin=0, vmax=1); a.set_title(f"+{n} implanted spots", fontsize=9)
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

rng, rows = np.random.default_rng(SEED), []
for n in [0, 2, 4, 6, 8, 10, 12]:
    Xi = np.stack([implant(X[i], n, rng) for i in never_te for _ in range(5)])
    rows.append({"implanted spots": n,
                 "boosting: P(smokes now)": predict_boosting(gb, Xi)[:, 2].mean(),
                 "CNN: P(smokes now)": predict_cnn(cnn, Xi)[:, 2].mean()})
impl = pd.DataFrame(rows).set_index("implanted spots")
display(impl.round(3))
impl.plot(marker="o", figsize=(6, 3.4), ylim=(0, 1), ylabel="mean P(smokes now)",
          title="Turning never-smokers into smokers, spot by spot"); plt.show()

If the CNN has learned the lesion pattern, its P(*smokes now*) should rise steadily with the number of implanted spots, crossing 0.5 roughly where real current smokers lie (8–12 spots). Boosting should react weakly and erratically, because new spots hit its few "important" pixels only by chance.

### 6.3 · Where does the CNN look? Class activation maps
Because the network ends with global average pooling and a linear layer, the score of class *c* is the average over positions of $\sum_k w_{c,k}\, A_k(x,y)$, where $A_k$ is filter map *k* of the last block and $w_{c,k}$ is its weight in the linear layer. Plotting $\sum_k w_{c,k} A_k(x,y)$ shows how much each position contributes (Zhou et al., 2016).

In [ ]:
def cam(model, img, c=2):
    with torch.no_grad():
        maps = model.features(torch.tensor(img[None, None], device=DEVICE))[0]
        m = torch.einsum("k,khw->hw", model.head.weight[c], maps).cpu().numpy()
    return ndi.zoom(m, 64 / m.shape[0], order=1)

show = [i for k in (2, 1, 0) for i in te[y[te] == k][:2]]
fig, axs = plt.subplots(2, 6, figsize=(13, 5))
for j, i in enumerate(show):
    p = predict_cnn(cnn, X[i:i + 1])[0]
    axs[0, j].imshow(X[i], cmap="gray", vmin=0, vmax=1)
    axs[0, j].set_title(f"true: {NAMES[CLASSES[y[i]]]}", fontsize=9, color=COLORS[CLASSES[y[i]]])
    axs[1, j].imshow(X[i], cmap="gray", vmin=0, vmax=1); axs[1, j].imshow(np.maximum(cam(cnn, X[i]), 0), cmap="hot", alpha=.55)
    axs[1, j].set_title(f"P(smokes now) = {p[2]:.2f}", fontsize=9)
for a in axs.ravel(): a.axis("off")
plt.suptitle("Bottom: positions that push the score of 'smokes now' up", fontsize=10); plt.tight_layout(); plt.show()

The heat should concentrate on the bright spots and streaks inside the lungs, wherever they occur, and not on fixed coordinates. Heatmaps are suggestive rather than proof, which is why experiments 6.1 and 6.2 test the same claim directly.

## 7 · Can boosting catch up with engineered features?

Boosting is strong on tabular data where **each column means something**. We can give it such columns by measuring the lesions ourselves:
1. Find the lungs (`lung_mask`).
2. Apply a **white top-hat** filter, which keeps bright structures smaller than about 7 pixels (spots, streaks) and removes the smooth background (body, heart, slowly varying exposure).
3. Summarise inside the lungs: number of spots, total and peak top-hat response, lung brightness and texture.

Each feature is defined relative to the lungs, so it is the same no matter where the chest sits in the image.

In [ ]:
def lesion_features(img):
    m = lung_mask(img)
    th = ndi.white_tophat(ndi.gaussian_filter(img, 0.8), size=7) * m
    spots, n_spots = ndi.label(th > 0.08)
    vals = img[m]
    return {"n_spots": n_spots,
            "tophat_sum": th.sum() / m.sum() * 1000,     # per 1,000 lung pixels
            "tophat_max": th.max(),
            "tophat_p99": np.percentile(th[m], 99),
            "bright_area": (th > 0.08).sum() / m.sum(),
            "lung_mean": vals.mean(), "lung_std": vals.std(), "lung_area": m.sum()}

FEAT = pd.DataFrame([lesion_features(im) for im in X])
FEAT["class"] = [CLASSES[k] for k in y]
display(FEAT.groupby("class").mean().loc[CLASSES].round(3))

fig, axs = plt.subplots(1, 3, figsize=(12, 3.2))
for a, f in zip(axs, ["n_spots", "tophat_sum", "lung_mean"]):
    a.boxplot([FEAT.loc[y == k, f] for k in range(3)]); a.set_title(f, fontsize=10)
    a.set_xticks([1, 2, 3], [NAMES[c] for c in CLASSES])
    a.tick_params(labelsize=8)
plt.tight_layout(); plt.show()

In [ ]:
F_np = FEAT.drop(columns="class").to_numpy(np.float32)
rows = []
for s, (tr_, te_) in enumerate(SPLITS):
    m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=7, min_samples_leaf=3,
                                       l2_regularization=1.0, random_state=SEED).fit(F_np[tr_], y[tr_])
    rows.append({"split": s, "model": "boosting · engineered features", **evaluate(y[te_], m.predict_proba(F_np[te_]))})
res_all = pd.concat([res, pd.DataFrame(rows)])
order_all = order + ["boosting · engineered features"]
display(res_all.groupby("model")[["accuracy", "macro_F1", "log_loss", "AUC"]].agg(["mean", "std"]).loc[order_all].round(3))

**Interpretation.** With eight meaningful columns instead of 4,096 pixels, boosting should jump close to the CNN, possibly matching it, at a fraction of the training cost. Spot count and top-hat response do almost all the work, and lung brightness alone adds little.

But consider what this required:
* **Domain knowledge.** We had to know that the signal is "number of small bright spots in the lungs". Here we know it because we know how the data were generated. With real radiographs, nobody can write down the full list of relevant patterns.
* **Hand-tuned image processing.** The lung mask threshold, the top-hat size and the spot threshold were all chosen by hand. They break when the image source changes (another hospital, another detector, a child's chest).
* **The CNN learned the equivalent of these features itself**, from 42 images, as Section 6 showed. That is the real argument for convolutional networks: they learn the representation, while boosting needs it handed over. In practice the two are combined: CNN-extracted features (embeddings) feed a boosted model together with tabular data such as age or BMI.

## 8 · May an insurer use such a model? (Switzerland)

*This is a discussion for the course, not legal advice.*

**What the rules say**
* **Health data are sensitive personal data** under the revised Federal Act on Data Protection (nFADP/revDSG, in force since 1 September 2023, Art. 5 lit. c). Where processing relies on consent, the consent must be **explicit** (Art. 6 para. 7).
* **Purpose limitation** (Art. 6 para. 3). The radiograph was taken for a medical reason. Reading it to infer smoking status is a new purpose, which the applicant must be told about and must be able to recognise. Consent to "share the image" is not automatically consent to "have the image scored for smoking".
* **Automated individual decisions** (Art. 21). If the model's output alone leads to a refusal or a premium loading, the insurer must inform the applicant and, on request, let them state their view and have a human review the decision.
* **Data protection impact assessment** (Art. 22). Profiling with sensitive data at scale is a typical high-risk case.
* **Duty of disclosure (Art. 4 VVG).** It covers the questions the insurer asks in writing. The insurer may already ask "Do you smoke?" directly. The model adds a way to *check* the answer, and with it the question whether contradicting a sworn answer with an imperfect model is fair (see below).
* **Medical secrecy (Art. 321 StGB).** The physician who holds the radiograph may only pass it on with the patient's consent or a legal basis.
* Basic insurance (KVG) must accept everyone, so such a model could only ever matter for **supplementary insurance (VVG)**.

**What the results say**
* **Error costs are asymmetric.** Labelling a never-smoker as a smoker can lead to a refusal or a higher premium and an implied accusation of lying (a breach of the duty of disclosure can allow the insurer to terminate the contract, Art. 6 VVG). Recall for *never* and the precision of *current* matter more than overall accuracy.
* **The middle class is the weakest.** Recent quitters are confused with never-smokers, so the model is least reliable exactly where an applicant's honest answer ("I quit 3 years ago") is hardest to verify.
* **Proxies and fairness.** In real images, lung markings also come from occupational dust, air pollution, past infections or age. A "smoking detector" may actually measure job or place of residence, which correlates with income and origin.
* **60 synthetic images prove nothing about real performance.** A real model would need large, representative data, external validation and calibration before anyone could consider using it.

**Our verdict.** Technically, a CNN can learn such patterns where boosting on raw pixels cannot. For underwriting, though, the **direct question plus the duty of disclosure** is the proportionate tool. Scoring medical images for a purpose other than the one they were taken for is hard to justify under purpose limitation and proportionality, would need explicit, informed consent and human review of every adverse decision, and carries a real risk of wrongly penalising honest applicants.

## 9 · Take-aways
1. **Representation matters more than the algorithm.** A table of pixels destroys neighbourhoods and position invariance; boosting then chases individual pixels.
2. **CNNs build in the right assumptions** for images: local filters, weight sharing (a spot is a spot anywhere) and pooling (count how many).
3. **With tiny data, evaluate honestly.** Use balanced splits, probability-based metrics (log-loss, AUC), and repeat over many splits.
4. **Boosting can compete when someone engineers the features**, but that shifts the hard part to human domain knowledge.
5. **Technically possible ≠ permitted.** Sensitive data, purpose limitation and the cost of false accusations dominate the business decision.

**References.** Kirchner, J. et al. (2012), *Br J Radiol* 85:339-345 (lung markings in smokers) · LeCun, Y. et al. (1998), *Proc. IEEE* 86:2278-2324 (CNNs) · Zhou, B. et al. (2016), *CVPR* (class activation maps) · Lu, M. T. et al. (2020), *Ann Intern Med* 173:704-713 (CNN on chest radiographs for lung-cancer risk).